In [1]:
import pandas as pd
from calendar import monthrange


In [2]:
# Function to transform the temperature data with date validation
def transform_temperature_data_validated(file_path):
    df = pd.read_csv(file_path)
    df_melted = df.melt(id_vars=["STATIONS", "ELEMENTS", "YEARS", "MONTHS"],
                        var_name="DAY",
                        value_name="TEMPERATURE")
    df_melted['MONTHS'] = df_melted['MONTHS'].apply(lambda x: f'{x:02d}')
    df_melted['DAY'] = df_melted['DAY'].str.extract('(\d+)')[0].astype(int).apply(lambda x: f'{x:02d}')
    
    # Create 'DATE' column and handle invalid dates
    dates = []
    for index, row in df_melted.iterrows():
        year, month, day = int(row['YEARS']), int(row['MONTHS']), int(row['DAY'])
        _, last_day_of_month = monthrange(year, month)
        if day <= last_day_of_month:
            dates.append(f"{year}-{month:02d}-{day:02d}")
        else:
            dates.append(None)  # None for invalid dates
    df_melted['DATE'] = pd.to_datetime(dates, errors='coerce')
    
    # Drop rows with NaT in 'DATE' and original columns
    df_final = df_melted.dropna(subset=['DATE']).drop(columns=["YEARS", "MONTHS", "DAY"])
    df_final_sorted = df_final.sort_values('DATE')
    df_final_sorted = df_final_sorted[['STATIONS', 'ELEMENTS', 'DATE', 'TEMPERATURE']]
    
    # Save to new CSV file
    output_path = file_path.replace('.csv', '_transformed_sorted.csv')
    df_final_sorted.to_csv(output_path, index=False)
    return output_path

# Function to transform the humidity data with datetime validation
def transform_humidity_data_with_datetime(file_path):
    df = pd.read_csv(file_path)
    df_melted = df.melt(id_vars=["Name", "Year", "Month", "Time"],
                        var_name="DAY",
                        value_name="HUMIDITY")
    df_melted['Month'] = df_melted['Month'].apply(lambda x: f'{x:02d}')
    df_melted['DAY'] = df_melted['DAY'].str.extract('(\d+)')[0].astype(int).apply(lambda x: f'{x:02d}')
    
    # Create 'DATETIME' column and handle invalid dates
    datetimes = []
    for index, row in df_melted.iterrows():
        year, month, day = int(row['Year']), int(row['Month']), int(row['DAY'])
        time = row['Time']
        _, last_day_of_month = monthrange(year, month)
        if day <= last_day_of_month:
            datetimes.append(f"{year}-{month:02d}-{day:02d} {time}")
        else:
            datetimes.append(None)  # None for invalid dates
    df_melted['DATETIME'] = pd.to_datetime(datetimes, errors='coerce')
    
    # Drop rows with NaT in 'DATETIME' and original columns
    df_final = df_melted.dropna(subset=['DATETIME']).drop(columns=["Year", "Month", "DAY", "Time"])
    df_final_sorted = df_final.sort_values('DATETIME')
    df_final_sorted = df_final_sorted[['Name', 'DATETIME', 'HUMIDITY']]
    
    # Save to new CSV file
    output_path = file_path.replace('.csv', '_transformed_sorted.csv')
    df_final_sorted.to_csv(output_path, index=False)
    return output_path

# Paths to the input files
min_temps_path = 'GMET_data/minimum_temperatures.csv'
max_temps_path = 'GMET_data/maximum_temperatures.csv'
humidity_path = 'GMET_data/relative_humidity.csv'

# Process the temperature and humidity files
min_temps_transformed_path = transform_temperature_data_validated(min_temps_path)
max_temps_transformed_path = transform_temperature_data_validated(max_temps_path)
humidity_transformed_path = transform_humidity_data_with_datetime(humidity_path)

print(f"Transformed minimum temperatures CSV: {min_temps_transformed_path}")
print(f"Transformed maximum temperatures CSV: {max_temps_transformed_path}")
print(f"Transformed humidity CSV: {humidity_transformed_path}")


Transformed minimum temperatures CSV: GMET_data/minimum_temperatures_transformed_sorted.csv
Transformed maximum temperatures CSV: GMET_data/maximum_temperatures_transformed_sorted.csv
Transformed humidity CSV: GMET_data/relative_humidity_transformed_sorted.csv
